In [1]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

GPU available: True
Device: Tesla T4


In [2]:
import os, zipfile, urllib.request

DATA_DIR = "/kaggle/working/tiny-imagenet-200"

if not os.path.exists(DATA_DIR):
    url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
    urllib.request.urlretrieve(url, "/kaggle/working/tiny-imagenet-200.zip")

    with zipfile.ZipFile("/kaggle/working/tiny-imagenet-200.zip", 'r') as zip_ref:
        zip_ref.extractall("/kaggle/working/")

In [3]:
import os, shutil

val_dir = os.path.join(DATA_DIR, "val")
images_dir = os.path.join(val_dir, "images")
annotations = os.path.join(val_dir, "val_annotations.txt")

with open(annotations) as f:
    for line in f:
        img, cls = line.split()[:2]
        os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
        shutil.move(
            os.path.join(images_dir, img),
            os.path.join(val_dir, cls, img)
        )

shutil.rmtree(images_dir)

In [4]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader
from torchvision.models import resnet50
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

In [5]:
def get_loaders(data_dir=DATA_DIR):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(64, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4802, 0.4481, 0.3975),
            (0.2302, 0.2265, 0.2262)
        ),
        transforms.RandomErasing(p=0.25)
    ])

    val_tf = transforms.Compose([
        transforms.Resize(64),
        transforms.CenterCrop(64),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4802, 0.4481, 0.3975),
            (0.2302, 0.2265, 0.2262)
        ),
    ])

    train_ds = torchvision.datasets.ImageFolder(
        os.path.join(data_dir, "train"),
        transform=train_tf
    )
    val_ds = torchvision.datasets.ImageFolder(
        os.path.join(data_dir, "val"),
        transform=val_tf
    )

    train_loader = DataLoader(
        train_ds, batch_size=128, shuffle=True,
        num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=256, shuffle=False,
        num_workers=0
    )

    return train_loader, val_loader

In [6]:
def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)

    cx, cy = np.random.randint(W), np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2


def cutmix(x, y, alpha=1.0):
    indices = torch.randperm(x.size(0)).to(x.device)
    lam = np.random.beta(alpha, alpha)

    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[indices, :, bbx1:bbx2, bby1:bby2]

    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size(-1) * x.size(-2)))

    return x, y, y[indices], lam

In [7]:
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    return 100 * correct / total

In [8]:
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, val_loader = get_loaders()

    model = resnet50(weights="IMAGENET1K_V1")
    model.fc = nn.Linear(model.fc.in_features, 200)
    model.to(device)

    optimizer = optim.SGD(
        model.parameters(), lr=0.1,
        momentum=0.9, weight_decay=1e-4
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = GradScaler()

    best_acc = 0
    patience = 20
    counter = 0

    for epoch in range(50):
        model.train()
        total_loss = 0

        for x, y in tqdm(train_loader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            use_cutmix = np.random.rand() < 0.3
            if use_cutmix:
                x, y_a, y_b, lam = cutmix(x, y)

            with autocast():
                out = model(x)
                if use_cutmix:
                    loss = lam * criterion(out, y_a) + (1 - lam) * criterion(out, y_b)
                else:
                    loss = criterion(out, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        val_acc = evaluate(model, val_loader, device)
        print(f"Epoch {epoch+1} | Loss {total_loss:.2f} | Val Acc {val_acc:.2f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print("Early stopping")
            break

        scheduler.step()

    print("Best Val Accuracy:", best_acc)

In [9]:
train()

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 135MB/s] 
/tmp/ipykernel_55/4226879360.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
  0%|          | 0/782 [00:00<?, ?it/s]/tmp/ipykernel_55/4226879360.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 782/782 [03:48<00:00,  3.42it/s]


Epoch 1 | Loss 3636.35 | Val Acc 16.58


100%|██████████| 782/782 [03:43<00:00,  3.50it/s]


Epoch 2 | Loss 3103.90 | Val Acc 25.60


...

Epoch 48 | Loss 1339.43 | Val Acc 59.27


100%|██████████| 782/782 [03:44<00:00,  3.48it/s]


Epoch 49 | Loss 1369.23 | Val Acc 59.33


100%|██████████| 782/782 [03:43<00:00,  3.50it/s]


Epoch 50 | Loss 1398.83 | Val Acc 59.52
Best Val Accuracy: 59.52


In [10]:
print(os.listdir("/kaggle/working"))

['tiny-imagenet-200.zip', 'best_model.pth', 'tiny-imagenet-200', '.virtual_documents']
